# Stage 2 and distilled upscaler inference

This notebook upscales a 512 × 512 input to 1024 × 1024 with:
- the stage 2 EMA teacher using 20 Euler steps and sigma shift 2.0;
- the round-1 distilled EMA student using 1, 2, and 4 uniform Euler steps.

All outputs use the same low-resolution latents, prompt, and initial noise. High-frequency
conditioning is disabled, so the models use their learned null embeddings.

Run from either the repository root or this notebook's directory. The checkpoint files and base
models must already exist at the paths below.

In [ ]:
import gc
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from safetensors.torch import load_file

from stdev.pretrain.config import SDXL_DIRECTORY, VAE_DIRECTORY
from stdev.text_encoders.sdxl_text_encoder import SDXLTextEncoder
from stdev.upscalers.pretrain_large import UpscalerUNet, sample_high_resolution_latents
from stdev.vaes.flux_2_vae import Flux2VAE

REPOSITORY_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()
)
STAGE2_WEIGHTS_PATH = (
    REPOSITORY_ROOT / "pretrain_runs/stage2_v1/export/stage2_final_000054108_ema.safetensors"
)
DISTILL_CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "distill_runs/distill_v1/checkpoints/checkpoint_r0001_000000613.pt"
)
IMAGE_PATH = REPOSITORY_ROOT / "datasets/4klsdb_subset/hr/103430.png"
OUTPUT_DIRECTORY = REPOSITORY_ROOT / "local_samples/inference"
PROMPT = "A duck."
SEED = 0
DEVICE = torch.device("cuda")

for checkpoint_path in (STAGE2_WEIGHTS_PATH, DISTILL_CHECKPOINT_PATH):
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    print(
        f"{checkpoint_path.relative_to(REPOSITORY_ROOT)}: {checkpoint_path.stat().st_size / 2**30:.2f} GiB"
    )

torch.backends.cudnn.benchmark = True

## Input image

The example takes a center square crop, resizes it to the stage 2 training resolution, and makes
the low-resolution input with bicubic downsampling. Replace `IMAGE_PATH` and `PROMPT` to experiment.

In [ ]:
def load_image_pair(
    image_path: Path, high_resolution_size: int = 1024
) -> tuple[np.ndarray, np.ndarray]:
    """
    returns:
        image_high: [H, W, 3], channel_order: RGB, pixel_values: [0...255], dtype: uint8
        image_low: [H / 2, W / 2, 3], channel_order: RGB, pixel_values: [0...255], dtype: uint8
    """
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read {image_path}")
    height, width = image.shape[:2]
    crop_size = min(height, width)
    top = (height - crop_size) // 2
    left = (width - crop_size) // 2
    image = image[top : top + crop_size, left : left + crop_size]
    image_high = cv2.resize(
        image, (high_resolution_size, high_resolution_size), interpolation=cv2.INTER_AREA
    )
    image_low = cv2.resize(
        image_high, (high_resolution_size // 2,) * 2, interpolation=cv2.INTER_CUBIC
    )
    return (
        cv2.cvtColor(image_high, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(image_low, cv2.COLOR_BGR2RGB),
    )


image_high, image_low = load_image_pair(IMAGE_PATH)
figure, axes = plt.subplots(1, 2, figsize=(10, 5))
for axis, image, title in zip(
    axes,
    (image_low, image_high),
    ("Low-resolution input (512 × 512)", "High-resolution reference (1024 × 1024)"),
    strict=True,
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()

## Encode the shared inputs

The text encoder is released after producing the pooled SDXL embedding. The VAE stays loaded for
decoding the generated high-resolution latents.

In [ ]:
vae = Flux2VAE(str(VAE_DIRECTORY), device=DEVICE, dtype=torch.bfloat16)
text_encoder = SDXLTextEncoder(str(SDXL_DIRECTORY), device=DEVICE, dtype=torch.float16)
text_embedding = text_encoder.encode_text_batch([PROMPT]).float().mean(dim=1)
del text_encoder
gc.collect()
torch.cuda.empty_cache()

image_low_tensor = (
    torch.from_numpy(image_low).permute(2, 0, 1)[None].to(DEVICE).float().div(255.0)
)  # [1, 3, 512, 512]
with torch.no_grad():
    latents_low = vae.encode(image_low_tensor)  # [1, 32, 64, 64]
del image_low_tensor

noise_generator = torch.Generator(device=DEVICE).manual_seed(SEED)
noise = torch.randn(
    (
        1,
        latents_low.shape[1],
        latents_low.shape[-2] * 2,
        latents_low.shape[-1] * 2,
    ),
    generator=noise_generator,
    device=DEVICE,
)  # [1, 32, 128, 128]
print(
    f"Low-resolution latents: {tuple(latents_low.shape)}, text embedding: {tuple(text_embedding.shape)}"
)

## Checkpoint loading and inference helpers

The distillation file contains raw weights, EMA weights, optimizer state, and training counters.
Only its EMA parameters are loaded here. `weights_only=True` avoids unrestricted pickle loading.

In [ ]:
def load_stage2_model(weights_path: Path) -> UpscalerUNet:
    model = UpscalerUNet()
    model.load_state_dict(load_file(str(weights_path), device="cpu"))
    return model.to(DEVICE, torch.bfloat16).eval().requires_grad_(False)


def load_distilled_model(checkpoint_path: Path) -> UpscalerUNet:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = UpscalerUNet()
    ema_weights = {
        name: ema_parameter
        for (name, _), ema_parameter in zip(
            model.named_parameters(), checkpoint["ema"], strict=True
        )
    }
    model.load_state_dict(ema_weights)
    print(f"Distillation state: {checkpoint['state']}")
    return model.to(DEVICE, torch.bfloat16).eval().requires_grad_(False)


@torch.no_grad()
def upscale_image(
    model: torch.nn.Module,
    num_steps: int,
    sigma_shift: float,
) -> tuple[np.ndarray, float]:
    """
    returns:
        image: [1024, 1024, 3], channel_order: RGB, pixel_values: [0...1], dtype: float32
        elapsed_seconds: wall-clock sampling and decoding time
    """
    batch_size = latents_low.shape[0]
    high_frequency_features = torch.zeros(batch_size, 4, device=DEVICE)
    high_frequency_mask = torch.zeros(batch_size, 4, dtype=torch.bool, device=DEVICE)
    text_mask = torch.ones(batch_size, dtype=torch.bool, device=DEVICE)
    torch.cuda.synchronize()
    start_time = time.perf_counter()
    latents_high = sample_high_resolution_latents(
        model,
        latents_low,
        text_embedding,
        text_mask,
        high_frequency_features,
        high_frequency_mask,
        num_steps=num_steps,
        sigma_shift=sigma_shift,
        noise=noise,
    )  # [1, 32, 128, 128]
    image = vae.decode(latents_high).float().clamp(0, 1)  # [1, 3, 1024, 1024]
    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - start_time
    return image[0].permute(1, 2, 0).cpu().numpy(), elapsed_seconds

## Stage 2 teacher: 20 steps

In [ ]:
stage2_model = load_stage2_model(STAGE2_WEIGHTS_PATH)
stage2_image, stage2_seconds = upscale_image(stage2_model, num_steps=20, sigma_shift=2.0)
print(f"Stage 2: {stage2_seconds:.1f}s")
del stage2_model
gc.collect()
torch.cuda.empty_cache()

## Distilled student: 1, 2, and 4 steps

In [ ]:
distilled_model = load_distilled_model(DISTILL_CHECKPOINT_PATH)
distilled_images: dict[int, np.ndarray] = {}
distilled_seconds: dict[int, float] = {}
for num_steps in (1, 2, 4):
    distilled_images[num_steps], distilled_seconds[num_steps] = upscale_image(
        distilled_model, num_steps=num_steps, sigma_shift=1.0
    )
    print(f"Distilled, {num_steps} step(s): {distilled_seconds[num_steps]:.1f}s")
del distilled_model
gc.collect()
torch.cuda.empty_cache()

## Compare and save

The high-resolution reference is only a visual target; it is not used by either model. Outputs
are saved as PNG files under `local_samples/inference/`.

In [ ]:
bicubic_image = (
    cv2.resize(image_low, (1024, 1024), interpolation=cv2.INTER_CUBIC).astype(np.float32) / 255.0
)
comparison = [
    ("Bicubic", bicubic_image),
    ("Stage 2 EMA · 20 steps", stage2_image),
    ("Distilled EMA · 1 step", distilled_images[1]),
    ("Distilled EMA · 2 steps", distilled_images[2]),
    ("Distilled EMA · 4 steps", distilled_images[4]),
    ("High-resolution reference", image_high.astype(np.float32) / 255.0),
]
figure, axes = plt.subplots(2, 3, figsize=(15, 10))
for axis, (title, image) in zip(axes.flat, comparison, strict=True):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
for title, image in comparison:
    file_name = title.lower().replace(" ", "_").replace("·", "").replace("__", "_") + ".png"
    plt.imsave(OUTPUT_DIRECTORY / file_name, np.clip(image, 0.0, 1.0))
figure.savefig(OUTPUT_DIRECTORY / "comparison.jpg", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved outputs to {OUTPUT_DIRECTORY}")